# Creación de Modelo MLP

In [ ]:
from src.utils.config import prices
from src.utils.files import read_file
from utils_modelo_precios.preprocesamiento import get_metrics

from sklearn.preprocessing import StandardScaler, PowerTransformer, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.inspection import permutation_importance
from sklearn.neural_network import MLPClassifier

from pandas import DataFrame, concat, Series

In [ ]:
df = read_file(prices)
df.head(2)

In [ ]:
df.columns

Como ya se ha estudiado en los notebooks de análisis, vamos a tener que transformar las variables numéricas. Debido a la distribución observada vamos a utilizar:

**Escalado logarítmico** (usando PowerTransform): num_languages, total_games_by_publisher, total_games_by_developer.

**StandardScaler**: description_len, release_year, brillo.

Las variables catergóricas ya están transformadas mediante multilabel encoding.

In [ ]:
def _preprocess_1(df):
    """Función para transformar los datos para realiza MLP

    Args:
        df (DataFrame): Datos iniciales del modelo que van a ser procesados.
    
    Returns:
        DataFrame: Datos que contienen las variables regresoras.
        DataFrame: Datos que contienen las variables respuesta.
    """
    df = df.drop(columns=['price_overview'])

    # Separación de DataFrames en diferentes tipos de variables
    y = DataFrame(df['price_range'])
    X_num_log = df[['num_languages', 'total_games_by_publisher', 'total_games_by_developer']]
    X_num_std = df[['description_len', 'release_year', 'brillo']]
    X_trans = df.drop(columns=['price_range', 'num_languages', 'total_games_by_publisher', 'total_games_by_developer', 'description_len', 'release_year', 'brillo'])

    # Transformación de variables
    pt = PowerTransformer(method='yeo-johnson')
    X_num_log_trans = pt.fit_transform(X_num_log)

    std = StandardScaler()
    X_num_std_trans = std.fit_transform(X_num_std)

    ohe = OneHotEncoder(sparse_output=False)
    y_trans = ohe.fit_transform(y)

    # Unificar datos transformados
    df_num_log_trans = DataFrame(X_num_log_trans, columns = pt.get_feature_names_out())
    df_num_std_trans = DataFrame(X_num_std_trans, columns = std.get_feature_names_out())
    df_y_trans = DataFrame(y_trans, columns = ohe.get_feature_names_out())

    df1 = concat([df_num_log_trans, df_num_std_trans], axis=1)
    df2 = concat([df1, X_trans], axis=1)

    display(df2.head(3))
    display(df_y_trans.head(3))

    return df2, df_y_trans

df_X, df_Y = _preprocess_1(df)

In [ ]:
# Vamos a deshechar las columnas de vectores de imágenes
df_X = df_X.drop(columns=['v_resnet', 'v_convnext', 'v_clip', 'id', 'name'], errors='ignore')

X_train, X_test, Y_train, Y_test = train_test_split(df_X, df_Y, test_size=0.3, random_state=42)
    
# Modelo base para medir cuánto baja el rendimiento del mismo al quitar variables de forma individual
mlp_base = MLPClassifier(hidden_layer_sizes=(30,30), max_iter=2000, random_state=42)
mlp_base.fit(X_train, Y_train)

test_importancia_variables = permutation_importance(
    mlp_base,
    X_test,
    Y_test,
    random_state=42,
    n_jobs=-1,
    n_repeats=10
)

In [ ]:
serie_ordenada_importancia = Series(test_importancia_variables.importances_mean, index=df_X.columns).sort_values(ascending=False)
serie_ordenada_importancia

Vamos a quitar las **variables menos importantes**:

Single-player                   0.000167

Steam Leaderboards              0.000083

Family Sharing                  0.000083

Co-op                          -0.000050

Strategy                       -0.000183

In [ ]:
df_X = df_X.drop(columns=['Single-player', 'Steam Leaderboards', 'Family Sharing', 'Co-op', 'Strategy'], errors='ignore')
df_X

In [ ]:
param_grid = {
    'hidden_layer_sizes': [(64,32), (100,), (32,16), (70,35)],
    'activation': ['relu', 'tanh'],
    'alpha': [0.0001, 0.01],
    'learning_rate_init': [0.001, 0.01]
}

grid = GridSearchCV(MLPClassifier(max_iter=5000, random_state=42), param_grid=param_grid, cv=5, n_jobs=-1)
grid.fit(X_test, Y_test)

In [ ]:
mlp_mejor_modelo = grid.best_estimator_
params_mejor_modelo = grid.best_params_
print(f'Los parámetros del mejor modelo son:\n{params_mejor_modelo}')

In [ ]:
Y_pred = mlp_mejor_modelo.predict(X_test)
get_metrics(Y_test, Y_pred, classes=list(Y_test.columns),  confusion_m=False)

In [ ]:
resultados_grid_df = DataFrame(grid.cv_results_)
display(resultados_grid_df)